# 02 · 비동기 처리와 실패 복구

동시성·용량 제한·대기시간을 각각 바꿉니다. 권장 20분.

위에서 아래로 실행합니다. 각 실험은 새 앱을 사용하며 실제 LLM·GPU·외부 API는 기본 경로에 없습니다. 먼저 결과를 예상하고 실행한 뒤 변수 하나를 바꾸고 관찰을 기록하세요.

In [ ]:
from pathlib import Path
import sys, asyncio, json, time
from dataclasses import replace
import numpy as np
import httpx
from IPython.display import display
ROOT = next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'course_labs').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from week09.lab.app import create_app
from week09.lab.service import Settings, ChatRequest, Service, TTLCache, TokenBucket, cache_key
from week09.lab.benchmark import run_load, summarize
assert sys.version_info[:2] == (3,11), 'Python 3.11 .venv 커널 선택'
OUT = ROOT/'output/validation/week09-lab'
OUT.mkdir(parents=True,exist_ok=True)
print('Python',sys.version.split()[0],'NumPy',np.__version__,'HTTPX',httpx.__version__)


## 기다리는 작업

asyncio.sleep은 네트워크 대기를 흉내 냅니다. CPU·GPU 연산이 빨라진다는 증거가 아닙니다.

In [ ]:
async def io_job():
    await asyncio.sleep(.03)
    return 'done'
start = time.perf_counter()
serial = []
for _ in range(4):
    serial.append(await io_job())
serial_s = time.perf_counter()-start
start = time.perf_counter()
parallel = await asyncio.gather(*(io_job() for _ in range(4)))
concurrent_s = time.perf_counter()-start
assert serial == parallel
display({'serial_s':serial_s,'concurrent_s':concurrent_s,'scope':'수면 모사'})

## 블로킹 호출의 분리

동기 I/O 함수는 필요 시 스레드로 이동할 수 있습니다. GIL이 있는 CPU Python 코드를 같은 이유로 병렬화하지 않습니다.

In [ ]:
def blocking_io():
    time.sleep(.01)
    return 'blocking I/O demo'
value = await asyncio.to_thread(blocking_io)
assert value == 'blocking I/O demo'

## 용량 한도

max_inflight=1이면 실행 중 요청 외의 요청을 즉시 503으로 거절합니다. 무제한 대기열이 없습니다.

In [ ]:
app = create_app(Settings(max_inflight=1,mock_delay_s=.08,timeout_s=.3))
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    first = asyncio.create_task(client.post('/api/v1/chat',json={'message':'A','use_cache':False}))
    while app.state.service.active == 0: await asyncio.sleep(.001)
    second = await client.post('/api/v1/chat',json={'message':'B'})
    original = await first
assert original.status_code == 200 and second.status_code == 503
assert app.state.service.active == 0
display(second.json())

## 제한시간 초과

asyncio.timeout이 상류 호출을 감쌉니다. 외부 GPU 작업의 종료 보장과는 다릅니다.

In [ ]:
app = create_app(Settings(timeout_s=.02))
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    timeout_response = await client.post('/api/v1/chat',json={'message':'x','scenario':'timeout'})
assert timeout_response.status_code == 504 and app.state.service.active == 0
display(timeout_response.json())

## 상류 오류

502와 504를 구분하고 원본 제공자 오류 본문을 노출하지 않습니다.

In [ ]:
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    upstream = await client.post('/api/v1/chat',json={'message':'x','scenario':'upstream_error'})
assert upstream.status_code == 502
display(upstream.json())

## 취소 후 자원 해제

클라이언트 취소 후 active가 남으면 후속 요청이 계속 거절될 수 있습니다.

In [ ]:
service = Service(Settings(timeout_s=1,mock_delay_s=.2))
task = asyncio.create_task(service.chat(ChatRequest(message='x'),'cancel-demo'))
await asyncio.sleep(.001)
task.cancel()
try:
    await task
except asyncio.CancelledError:
    pass
assert service.active == 0
display(list(service.traces))

## 빈도 제한과 용량 제한

Token bucket은 시간당 유입, active는 실행 중 작업 수입니다. 이 예시는 전체 프로세스 버킷 하나입니다.

In [ ]:
now = [0.]
bucket = TokenBucket(2,2,clock=lambda:now[0])
assert [bucket.allow() for _ in range(3)] == [True,True,False]
now[0] = .5
assert bucket.allow()
print('0.5초 후 토큰 1개 보충')

## 429 응답

캐시 적중 요청도 유입 한도를 소비합니다.

In [ ]:
app = create_app(Settings(burst=1,rate_per_s=.001))
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    one = await client.post('/api/v1/chat',json={'message':'x'})
    two = await client.post('/api/v1/chat',json={'message':'x'})
assert one.status_code == 200 and two.status_code == 429
display(two.headers.get('retry-after'))

## 재시도의 조건

이번 앱은 자동 재시도를 하지 않습니다. 생성 중복·과금·증폭 위험과 서버 안내를 먼저 검토합니다.

In [ ]:
retry_plan = {'422':'재시도 전 입력 수정','429':'Retry-After·전체 시간 예산','503':'백오프·횟수 상한·용량 확인','504':'상류 작업 생존 여부·중복 비용 확인'}
assert len(retry_plan) == 4
display(retry_plan)

## 실패 복구 기록

실패 코드를 지우고 정상값으로 바꾸지 않습니다.

In [ ]:
report = {'observed':[503,504,502,429],'resource_release':'active=0 확인','remaining':'실제 제공자 취소·다중 worker·인증 미검증'}
(OUT/'reliability.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
display(report)

## 포트폴리오 기록

변경 변수 / 고정 조건 / 실제 관측값 / 실패 사례 / 해석의 한계 / 다음 확인을 기록하세요. 개인정보·학생 원문은 공개 저장소에 넣지 않습니다. 별도 과제 마감은 소속 공지를 따르며 이 실습에서 새 제출 기한을 만들지 않습니다.